# Creating and configuring a Pinecone index

To kickstart your semantic search application,
you'll create and configure a new Pinecone index named 'pinecone-datacamp'.
You'll use this in subsequent exercises to host Wikipedia articles from the SQuAD dataset.

In [20]:
from pinecone import ServerlessSpec
from pinecone_helper import get_pinecone_client

pc = get_pinecone_client()
index_name = "pinecone-datacamp"

need_to_delete_indexes = [index_name]

pinecone_indexes = pc.list_indexes()

for to_delete_index in need_to_delete_indexes:
    if to_delete_index in [index.name for index in pinecone_indexes]:
        pc.delete_index(to_delete_index)
        print(f"{to_delete_index} successfully deleted.")
    else:
        print(f"{to_delete_index} not in index list.")

# Create Pinecone index
pc.create_index(
    name=index_name,
    dimension=1536,
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1",
    ),
)

# Connect to index and print the index statistics
index = pc.index(index_name)
print(index.describe_index_stats())

pinecone-datacamp successfully deleted.
DescribeIndexStatsResponse(dimension=1536, total_vector_count=0, metric='cosine', namespaces=0)


# Upserting vectors for semantic search

Time to embed some text data and upsert the vectors
and metadata into your 'pinecone-datacamp' index!
You've be given a dataset named squad_dataset.csv,
and a sample of 200 rows has been loaded in the DataFrame, df.

```python
print(df.head())
                         id                                               text                     title
0  5733be284776f41900661182  Architecturally, the school has a Catholic cha...  University_of_Notre_Dame
1  5733bf84d058e614000b61be  As at most other universities, Notre Dame's st...  University_of_Notre_Dame
2  5733bed24776f41900661188  The university is the major seat of the Congre...  University_of_Notre_Dame
3  5733a6424776f41900660f51  The College of Engineering was established in ...  University_of_Notre_Dame
4  5733a70c4776f41900660f64  All of Notre Dame's undergraduate students are...  University_of_Notre_Dame

print(df.info())
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   id      200 non-null    object
 1   text    200 non-null    object
 2   title   200 non-null    object
dtypes: object(3)
memory usage: 4.8+ KB
None
```

In this exercise, to interact with the OpenAI API to use their embedding model.

Your task is to embed the text using OpenAI's API and upsert the embeddings
and metadata into the Pinecone index under the namespace, squad_dataset.

In [21]:
import os

# from uuid import uuid4
import pandas as pd
from file_helper import get_file_dir
from openai_helper import create_embeddings
from pinecone_helper import get_index

csv_path = os.path.join(get_file_dir(), "squad_dataset.csv")
df = pd.read_csv(csv_path, nrows=200)

In [22]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   id      200 non-null    str  
 1   text    200 non-null    str  
 2   title   200 non-null    str  
dtypes: str(3)
memory usage: 4.8 KB


In [23]:
df.head()

,id,text,title
0,5733be284776f41900661182,"Architecturally, the school has a Catholic cha...",University_of_Notre_Dame
1,5733bf84d058e614000b61be,"As at most other universities, Notre Dame's st...",University_of_Notre_Dame
2,5733bed24776f41900661188,The university is the major seat of the Congre...,University_of_Notre_Dame
3,5733a6424776f41900660f51,The College of Engineering was established in ...,University_of_Notre_Dame
4,5733a70c4776f41900660f64,All of Notre Dame's undergraduate students are...,University_of_Notre_Dame


In [25]:
index = get_index(index_name)
batch_limit = 100
namespace = "squad_dataset"

for start in range(0, len(df), batch_limit):
    batch = df.iloc[start : start + batch_limit]

    # Extract the metadata form each row
    metadatas = [
        {"text_id": row["id"], "text": row["text"], "title": row["title"]}
        for _, row in batch.iterrows()
    ]
    texts = batch["text"].tolist()

    # ids = [str(uuid4()) for _ in range(len(texts))]
    # Use the original dataset ID as the Pinecone vector ID
    ids = batch["id"].astype(str).tolist()

    # Encode texts using OpenAI
    embeds = create_embeddings(texts)

    # Create Pinecone vectors
    vectors = [
        (id_, embedding, metadata)
        for id_, embedding, metadata in zip(ids, embeds, metadatas)
    ]

    # Upsert vectors to the correct namespace
    index.upsert(vectors=vectors, namespace=namespace)

# Print the index statistics
print(index.describe_index_stats())

DescribeIndexStatsResponse(dimension=1536, total_vector_count=200, metric='cosine', namespaces=1)


# Querying vectors for semantic search

In this exercise, you'll create a query vector from the question,
'What is in front of the Notre Dame Main Building?'.
Using this embedded query, you'll query the 'squad_dataset' namespace
from the 'pinecone-datacamp' index and return the top five most similar vectors.

In [26]:
from openai_helper import create_embeddings
from pinecone_helper import get_index

index = get_index(index_name)
query = "What is in front of the Notre Dame Main Building?"

# Create the query vector
query_emb = create_embeddings(query)

# Query the index and retrieve the top five most similar vectors
retrieved_docs = index.query(
    vector=query_emb, top_k=5, namespace=namespace, include_metadata=True
)

for result in retrieved_docs["matches"]:
    print(f"{result['id']}: {round(result['score'], 2)}")
    print("\n")

5733ad384776f41900660fec: 0.54


5733be284776f41900661182: 0.53


57339a5bd058e614000b5e91: 0.52


573382a14776f41900660c2d: 0.51


573383e94776f41900660c5a: 0.48


